<div dir="rtl">

# 04 - Transform: تنظيف داتا الموبايلات

ده Notebook مرحلة الـ **Transform** في الـ ETL Pipeline. بياخد الداتا الخام اللي سحبتها من Jumia في `01_extract_jumia`، وينضفها ويجهزها للتحليل والتخزين في SQL.

**المشاكل اللي لقيتها في الداتا الخام لما فحصتها:**

| المشكلة | مثال |
|---|---|
| الأسعار مكتوبة كنص مش أرقام | `EGP 8,599.00` |
| منتجات مش موبايلات في القسم | تكييفات، غلايات، ميكروويف، جرابات |
| نفس الماركة مكتوبة بأكتر من شكل | `XIAOMI` و `Mi` و `Redmi`، و `Infinx` بدل `Infinix` |
| الموبايلات المستعملة ماركتها `Renewed` | `Renewed iPhone 13 Pro ...` |
| التقييم `0` للمنتج اللي محدش قيّمه | 1,456 منتج تقييمهم 0 ومن غير ولا review |
| المساحة والرام جوه الاسم بأشكال كتير | `128GB/4GB`، `4GB - 64GB`، `8GB RAM 256GB ROM`، `(RAM:4,ROM:64)` |
| مسافات زيادة في الأسماء | `Samsung  Galaxy`، `Infinx  Smart 20` |

**قاعدة مهمة:** مش بعدل على ملف الـ raw نفسه. بقرا منه وبحفظ النسخة النضيفة في `data/clean`.

</div>

<div dir="rtl">

## الخطوة 1: استدعاء المكتبات

| المكتبة | استخدمتها في إيه |
|---|---|
| `pandas` | قراية الداتا وتنظيفها وحفظها |
| `re` | البحث جوه النص بـ Regular Expressions، عشان أطلع المساحة والرام من الاسم |
| `os` | عشان ألاقي آخر ملف raw وأعمل فولدر `data/clean` |

</div>

In [1]:
import pandas as pd
import re
import os

<div dir="rtl">

## الخطوة 2: قراية آخر ملف raw

بما إني بسحب كل يوم ملف جديد، بجيب أسماء كل الملفات اللي في `data/raw` وبختار آخر ملف CSV لـ Jumia.

الملفات اسمها فيه التاريخ بالشكل ده `2026-10-24`، فلما أرتبها بـ `sorted` بتترتب بالتاريخ، وآخر واحد في الـ list هو الأحدث.

</div>

In [2]:
raw_files = []

for file_name in os.listdir("data/raw"):
    if file_name.startswith("jumia_raw_") and file_name.endswith(".csv"):
        raw_files.append(file_name)

raw_files = sorted(raw_files)
raw_file = raw_files[-1]

df = pd.read_csv("data/raw/" + raw_file)

print("File:", raw_file)
print("Shape:", df.shape)

File: jumia_raw_2026-10-24.csv
Shape: (1848, 18)


<div dir="rtl">

## الخطوة 3: تنضيف أسماء المنتجات

- `.str.replace(r"\s+", " ", regex=True)` بتحول أي مسافات ورا بعض لمسافة واحدة. `\s+` معناها "مسافة واحدة أو أكتر".
- `.str.strip()` بتشيل المسافات من أول وآخر الاسم.

</div>

In [3]:
df["product_name"] = df["product_name"].str.replace(r"\s+", " ", regex=True)
df["product_name"] = df["product_name"].str.strip()

df["product_name"].head()

0    Honor X5c Plus Dual SIM 4G 128GB/4GB - Midnigh...
1    Honor X5c Plus Dual SIM 4G 128GB/4GB - Ocean Cyan
2    Honor T10 AI Translator Phone 64GB 4GB , 4G Du...
3    OPPO A6 -6.7-Inch 128GB/4GB Dual SIM 4G Mobile...
4    Samsung Samsung Galaxy A16 Dual SIM, 128GB, 4G...
Name: product_name, dtype: str

<div dir="rtl">

## الخطوة 4: تحويل الأسعار والخصم لأرقام

السعر جاي كنص زي `EGP 8,599.00`، وبالشكل ده مقدرش أحسب متوسط أو أقارن بين سعرين. عشان كده:

1. بشيل `EGP ` و الفاصلة `,`.
2. بحول النص لرقم بـ `.astype(float)`.

ونفس الفكرة مع الخصم: `9%` بقى `9`.

المنتج اللي مفيهوش خصم بيفضل `old_price` و `discount` بتوعه فاضيين (`NaN`)، وده صح لأن فعلاً مالوش سعر قديم.

</div>

In [4]:
df["price"] = df["price"].str.replace("EGP ", "").str.replace(",", "").astype(float)
df["old_price"] = df["old_price"].str.replace("EGP ", "").str.replace(",", "").astype(float)
df["discount"] = df["discount"].str.replace("%", "").astype(float)

df[["price", "old_price", "discount"]].head()

,price,old_price,discount
0,8599.0,9460.0,9.0
1,8599.0,9460.0,9.0
2,6975.0,7699.0,9.0
3,12525.0,13780.0,9.0
4,11599.0,12717.0,9.0


<div dir="rtl">

## الخطوة 5: التقييم 0 معناه "مفيش تقييم"

المنتج اللي محدش قيّمه Jumia بتكتبله تقييم `0`. لو سبتها كده، متوسط تقييم الماركة هينزل غلط. يعني لو ماركة عندها 10 موبايلات و 9 منهم محدش قيّمهم، متوسطها هيطلع قريب من صفر وهو مش كده.

عشان كده بحول التقييم لـ `None` (فاضي) لما يكون `reviews_count` بـ 0. و pandas بتتجاهل الخانات الفاضية لما تحسب المتوسط.

`df.loc[الشرط, "العمود"] = القيمة` معناها: غير قيمة العمود ده في الصفوف اللي الشرط فيها صح بس.

</div>

In [5]:
print("Rating = 0 before:", (df["rating"] == 0).sum())

df.loc[df["reviews_count"] == 0, "rating"] = None

print("Rating = 0 after:", (df["rating"] == 0).sum())
print("Products with no rating:", df["rating"].isnull().sum())

Rating = 0 before: 1456
Rating = 0 after: 0
Products with no rating: 1456


<div dir="rtl">

## الخطوة 6: شيل المنتجات اللي مش موبايلات

لما رتبت الداتا بالماركة، لقيت جوه قسم Smartphones تكييفات Haier و Carrier، وغلايات، وميكروويف Toshiba، وجرابات موبايل.

عملت list بالكلمات اللي بتدل على إن المنتج مش موبايل، وبلف على كل اسم: لو فيه أي كلمة منهم يبقى مش موبايل.

**ليه `"back cover"` مش `"cover"` بس؟** لأن فيه موبايلات بتيجي معاها هدية زي `+ Free Grey Cover`، ودي موبايلات حقيقية مش عايز أشيلها.

قبل ما أشيلهم بطبعهم، عشان أتأكد إني مش بشيل موبايل بالغلط.

</div>

In [6]:
not_phone_words = [
    "air conditioner", "air cooler", "air fryer", "kettle", "microwave",
    "blender", "stand fan", "ventilating", "coffee grinder", "racing wheel",
    "back cover",
]

is_phone_list = []

for name in df["product_name"]:
    name_lower = name.lower()
    is_phone = True
    for word in not_phone_words:
        if word in name_lower:
            is_phone = False
    is_phone_list.append(is_phone)

df["is_phone"] = is_phone_list

removed = df[df["is_phone"] == False]
print("Not phones:", len(removed))
removed[["brand", "product_name", "price"]]

Not phones: 19


,brand,product_name,price
267,Fresh,Fresh Kitchen Ventilating 6 Inch 15 CM - White...,669.0
1008,Generic,Back Cover For Samsung Galaxy A17 Case with Ca...,249.0
1143,General,General Back Cover Carbon With Magnetic For Sa...,799.0
1401,Carrier,Carrier Optimax Air Conditioner 4 HP Split Coo...,83999.0
1573,Mienta,Mienta Blender 500 Watt Capacity 1.5 L - White...,2399.0
1574,Media Tech,Media Tech MediaTech Stand Fan 18 Inch - 6 Bla...,2299.0
1575,Haier,Haier Air Conditioner 1.5 HP Smart ULTRA Inver...,39399.0
1576,Media Tech,Media Tech Air Cooler 80 Liter - 250 Watt Digi...,7499.0
1577,Media Tech,Media Tech Air Cooler 90 Liter - 250 Watt - Bl...,6499.0
1578,TORNADO,TORNADO Air Fryer 1800 Watt - 8 Liter Manual C...,6699.0


In [7]:
df = df[df["is_phone"] == True]
df = df.drop(columns=["is_phone"])

print("Shape after removing:", df.shape)

Shape after removing: (1829, 18)


<div dir="rtl">

## الخطوة 7: توحيد أسماء الماركات

عملت dictionary فيه الاسم الغلط والاسم الصح:
- `XIAOMI` و `Mi` و `Redmi` كلهم **Xiaomi**. Redmi و Poco ماركات تابعة لـ Xiaomi.
- `Infinx` غلطة كتابة، والصح **Infinix**.
- `Hmd` الصح **HMD**.

**الموبايلات المستعملة:** Jumia كاتبة الماركة بتاعتهم `Renewed`، وده مش اسم ماركة. فبعمل عمود جديد `is_renewed` بـ True ليهم، وبطلع الماركة الحقيقية من الاسم. كلهم iPhone، يعني **Apple**.

`brand_fix.get(brand, brand)` معناها: لو الماركة موجودة في الـ dictionary هات الاسم الصح، ولو مش موجودة سيبها زي ما هي.

</div>

In [8]:
brand_fix = {
    "XIAOMI": "Xiaomi",
    "Mi": "Xiaomi",
    "Redmi": "Xiaomi",
    "Infinx": "Infinix",
    "Hmd": "HMD",
}

brand_list = []
is_renewed_list = []

for brand, name in zip(df["brand"], df["product_name"]):

    if brand == "Renewed":
        is_renewed_list.append(True)
        if "iphone" in name.lower():
            brand = "Apple"
        else:
            brand = None
    else:
        is_renewed_list.append(False)

    brand_list.append(brand_fix.get(brand, brand))

df["brand"] = brand_list
df["is_renewed"] = is_renewed_list

df["brand"].value_counts().head(12)

brand
Samsung     459
Honor       237
Xiaomi      237
Infinix     201
OPPO        177
Apple       162
realme      157
vivo         95
Tecno        49
Huawei       16
Itel         12
Motorola      6
Name: count, dtype: int64

<div dir="rtl">

## الخطوة 8: استخراج المساحة والرام من الاسم

دي أصعب خطوة، لأن كل بائع كاتب الاسم بطريقته:

| شكل الاسم | المساحة | الرام |
|---|---|---|
| `Samsung Galaxy A16 128GB/6GB` | 128 | 6 |
| `XIAOMI Redmi A7 Pro 4GB - 64GB` | 64 | 4 |
| `Samsung MOBILE GALAXY A16 8GB RAM 256GB ROM` | 256 | 8 |
| `Infinix Smart 20 (RAM:4,ROM:64)` | 64 | 4 |
| `Samsung A26 8/256` | 256 | 8 |
| `Apple iPhone 18 Pro Max 1TB` | 1024 | فاضي |

**الفكرة:** المساحة والرام ليهم أرقام معروفة. المساحة بتبقى 32 أو 64 أو 128 أو 256 أو 512 أو 1024. والرام بتبقى من 1 لـ 16. فبطلع كل الأرقام اللي جنبها `GB` أو `TB`، وأشوف كل رقم ينفع يبقى إيه.

**الخطوات لكل اسم:**
1. `re.findall(r"(\d+)\s*(tb|gb|gega)", text)` بتجيب كل رقم جنبه وحدة. `\d+` معناها رقم أو أكتر، و `\s*` معناها مسافة أو مفيش.
2. أكبر رقم موجود في `storage_sizes` هو المساحة.
3. أول رقم موجود في `ram_sizes` (ومش هو المساحة) هو الرام.
4. لو لسه ملقيتش، بدور على أشكال تانية: `RAM:4` و `ROM:64` و `8/256` و `8+256`.

**الموبايلات اللي مساحتها بالـ MB** (موبايلات زراير قديمة) بسيبها فاضية، لأن الأرقام فيها مش GB.

**iPhone:** Apple مش بتكتب الرام في اسم الموبايل، فالرام بتاعته هتفضل فاضية، وده طبيعي.

</div>

In [9]:
storage_sizes = [32, 64, 128, 256, 512, 1024, 2048]
ram_sizes = [1, 2, 3, 4, 6, 8, 12, 16]

storage_list = []
ram_list = []

for name in df["product_name"]:
    text = name.lower()
    storage = None
    ram = None

    if "mb" not in text:

        # 1) Numbers with GB / TB
        sizes = re.findall(r"(\d+)\s*(tb|gb|gega)", text)

        for number, unit in sizes:
            size = int(number)
            if unit == "tb":
                size = size * 1024
            if size in storage_sizes:
                if storage is None or size > storage:
                    storage = size

        for number, unit in sizes:
            size = int(number)
            if size in ram_sizes and size != storage and ram is None:
                ram = size

        # 2) Other shapes: "8G RAM", "RAM:4", "ROM:64", "8/256", "8+256"
        match = re.search(r"(\d+)\s*g?\s*ram", text)
        if ram is None and match and int(match.group(1)) in ram_sizes:
            ram = int(match.group(1))

        match = re.search(r"ram\s*:?\s*(\d+)", text)
        if ram is None and match and int(match.group(1)) in ram_sizes:
            ram = int(match.group(1))

        match = re.search(r"rom\s*:?\s*(\d+)", text)
        if storage is None and match and int(match.group(1)) in storage_sizes:
            storage = int(match.group(1))

        match = re.search(r"(\d+)\s*g?b?\s*[/+]\s*(\d+)", text)
        if match:
            first = int(match.group(1))
            second = int(match.group(2))
            if storage is None and max(first, second) in storage_sizes:
                storage = max(first, second)
            if ram is None and min(first, second) in ram_sizes:
                ram = min(first, second)

        # 3) Old phones with 16GB storage
        if storage is None and "16gb" in text.replace(" ", ""):
            storage = 16

    storage_list.append(storage)
    ram_list.append(ram)

df["storage_gb"] = storage_list
df["ram_gb"] = ram_list

df[["product_name", "storage_gb", "ram_gb"]].sample(10, random_state=1)

,product_name,storage_gb,ram_gb
258,"Honor X5c Plus Dual SIM, 128GB, 4GB RAM, 4G LT...",128.0,4.0
988,Samsung Galaxy A07 Dual SIM 4G 6GB - 128GB - B...,128.0,6.0
926,Infinix Hot 70 Dual SIM 4G 6GB - 128GB - Therm...,128.0,6.0
352,Samsung Galaxy A07 4GB-128GB Mobile Phone Dual...,128.0,4.0
231,"Huawei nova 15 Max , 256 GB , 8 GB , 4G , Dual...",256.0,8.0
1464,"vivo Y21 Dual SIM, 128GB, 4GB RAM, 5G - BLACK",128.0,4.0
1779,XIAOMI Redmi A3x Dual SIM 4G 128GB/4GB Mobile ...,128.0,4.0
694,vivo Y05 64GB - 4GB Ram - Light Platinum - 4G,64.0,4.0
603,OPPO RENO 15F 12GB/ 256GB 5G -AURORA BLUE,256.0,12.0
1522,"Infinix Smart 20 Dual SIM, 64GB, 4GB RAM, 4G L...",64.0,4.0


<div dir="rtl">

**بتأكد من النتيجة:** ببص على الأرقام اللي طلعت، وعلى المنتجات اللي ملقيتش فيها مساحة أو رام (غير Apple)، عشان أعرف القاعدة فاتها إيه.

</div>

In [10]:
print("Storage values:")
print(df["storage_gb"].value_counts())
print()
print("RAM values:")
print(df["ram_gb"].value_counts())

Storage values:
storage_gb
256.0     927
128.0     549
64.0      205
512.0     111
16.0        7
1024.0      7
32.0        5
Name: count, dtype: int64

RAM values:
ram_gb
8.0     575
4.0     430
12.0    330
6.0     269
3.0      45
16.0      9
2.0       8
1.0       6
Name: count, dtype: int64


In [11]:
not_apple = df[df["brand"] != "Apple"]
missing = not_apple[not_apple["storage_gb"].isnull() | not_apple["ram_gb"].isnull()]

print("Not Apple and missing storage or RAM:", len(missing))
missing[["product_name", "storage_gb", "ram_gb"]]

Not Apple and missing storage or RAM: 24


,product_name,storage_gb,ram_gb
97,"Hmd 150 Music, Dual SIM Mobile - TA-1709 – DAR...",NaN,NaN
125,Nokia Afrutto - BM10 Dual Sim Mobile Phone Min...,NaN,NaN
515,OPPO reno 15f,NaN,NaN
798,vivo Y05e 4G-64GB - Teal,64.0,NaN
839,XIAOMI Note 15,NaN,NaN
1006,Sunelan SunelanS8 Dual SIM 32MB 32MB RAM 2G - ...,NaN,NaN
1224,XIAOMI Note 15,NaN,NaN
1237,OPPO OPPORENO 15F,NaN,NaN
1424,Sunelan SunelanS550 Dual SIM 64MB 32MB RAM 2G ...,NaN,NaN
1457,"realme C100 Dual Sim ,256GB, 8BG Ram,4G - White",256.0,NaN


<div dir="rtl">

الباقيين دول أسماءهم ناقصة فعلاً (زي `XIAOMI Note 15` من غير أي مساحة)، أو موبايلات زراير، أو فيهم غلطة كتابة (زي `2568GB` بدل `256GB`، و `8BG` بدل `8GB`). مش هخمّن قيمتهم، وهسيبهم فاضيين. **الخانة الفاضية أحسن من رقم غلط.**

</div>

<div dir="rtl">

## الخطوة 9: نوع الشبكة (5G / 4G)

بلف على الأسماء: لو فيها `5g` يبقى 5G. لو فيها `4g` أو `lte` يبقى 4G. ولو فيها `3g` أو `2g` بكتبهم زي ما هما.

بدور على 5G الأول، لأن فيه موبايلات 5G مكتوب في اسمها `4G` كمان، زي `Dual SIM 4G` في وصف الشريحة.

**ليه `re.search(r"\b5g\b", ...)` مش `"5g" in name`؟** لأن `\b` معناها حدود الكلمة. من غيرها، `"5g" in "a35g..."` ممكن تطلع True بالغلط من جوه كلمة تانية.

حوالي 155 منتج هيفضل نوع الشبكة بتاعهم فاضي، لأن البائع مكتبهوش في الاسم أصلاً (زي `Samsung A26 8/256`).

</div>

In [12]:
network_list = []

for name in df["product_name"]:
    text = name.lower()

    if re.search(r"\b5g\b", text):
        network = "5G"
    elif re.search(r"\b4g\b", text) or "lte" in text:
        network = "4G"
    elif re.search(r"\b3g\b", text):
        network = "3G"
    elif re.search(r"\b2g\b", text):
        network = "2G"
    else:
        network = None

    network_list.append(network)

df["network"] = network_list

df["network"].value_counts(dropna=False)

network
4G     920
5G     749
NaN    155
2G       4
3G       1
Name: count, dtype: int64

<div dir="rtl">

## الخطوة 10: الموبايلات اللي معاها هدية

فيه موبايلات (خصوصاً Honor) بتتباع ومعاها سماعة أو ساعة هدية، زي `+ Free Earbuds X7 Lite`. السعر هنا مش سعر الموبايل لوحده، فلو قارنته بسعر نفس الموبايل في موقع تاني من غير هدية، المقارنة هتبقى ظالمة.

عشان كده عملت عمود `has_gift`، وفي مرحلة المقارنة أقدر أفصلهم.

</div>

In [13]:
gift_words = ["free", "earbuds", "watch", "gift", "headset"]

has_gift_list = []

for name in df["product_name"]:
    name_lower = name.lower()
    has_gift = False
    for word in gift_words:
        if word in name_lower:
            has_gift = True
    has_gift_list.append(has_gift)

df["has_gift"] = has_gift_list

print("Products with a gift:", df["has_gift"].sum())
df[df["has_gift"] == True][["product_name", "price"]].head()

Products with a gift: 22


,product_name,price
607,"Honor 600 Lite 5G Smartphone, 8GB RAM + 256GB ...",21999.0
1678,Honor 400 Dual SIM 5G Smartphone - 256GB/12GB ...,26648.7
1686,Honor 400 Dual SIM 5G Smartphone - 512GB/12GB ...,25999.0
1693,Nothing 3A - Dual SIM 5G 128GB/8GB Mobile Phon...,23998.8
1709,Honor 400 Dual SIM 5G Smartphone - 256GB/12GB ...,26648.7


<div dir="rtl">

## الخطوة 11: ترتيب الأعمدة وشيل التكرار

- شلت عمود `page`، لأنه كان مهم وقت السحب بس.
- رتبت الأعمدة بحيث المهم يبقى في الأول.
- `drop_duplicates(subset=["sku"])` بتشيل أي منتج متكرر بنفس الـ SKU لو موجود.
- `reset_index(drop=True)` بترجع ترقيم الصفوف من 0 بعد ما شلت صفوف.

</div>

In [14]:
df = df[[
    "site", "sku", "product_name", "brand", "category",
    "storage_gb", "ram_gb", "network",
    "price", "old_price", "discount",
    "rating", "reviews_count",
    "is_official_store", "is_express", "is_sponsored", "is_renewed", "has_gift",
    "seller_id", "url", "image_url", "scraped_at",
]]

df = df.drop_duplicates(subset=["sku"])
df = df.reset_index(drop=True)

print("Shape:", df.shape)
df.head()

Shape: (1829, 22)


,site,sku,product_name,brand,category,storage_gb,ram_gb,network,price,old_price,...,reviews_count,is_official_store,is_express,is_sponsored,is_renewed,has_gift,seller_id,url,image_url,scraped_at
0,jumia,HO474MP26BRG1NAFAMZ,Honor X5c Plus Dual SIM 4G 128GB/4GB - Midnigh...,Honor,Android Phones,128.0,4.0,4G,8599.0,9460.0,...,12,False,True,False,False,False,NaN,https://www.jumia.com.eg/honor-x5c-plus-dual-s...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,2026-10-24 07:16:54
1,jumia,HO474MP2CA3HTNAFAMZ,Honor X5c Plus Dual SIM 4G 128GB/4GB - Ocean Cyan,Honor,Android Phones,128.0,4.0,4G,8599.0,9460.0,...,20,False,True,False,False,False,NaN,https://www.jumia.com.eg/honor-x5c-plus-dual-s...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,2026-10-24 07:16:54
2,jumia,HO474MP27U60XNAFAMZ,"Honor T10 AI Translator Phone 64GB 4GB , 4G Du...",Honor,Android Phones,64.0,4.0,4G,6975.0,7699.0,...,1,False,True,False,False,False,201886.0,https://www.jumia.com.eg/honor-t10-ai-translat...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,2026-10-24 07:16:54
3,jumia,OP872MP1WHCI3NAFAMZ,OPPO A6 -6.7-Inch 128GB/4GB Dual SIM 4G Mobile...,OPPO,Android Phones,128.0,4.0,4G,12525.0,13780.0,...,7,False,True,False,False,False,NaN,https://www.jumia.com.eg/oppo-a6-6.7-inch-128g...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,2026-10-24 07:16:54
4,jumia,SA024MP220DBFNAFAMZ,"Samsung Samsung Galaxy A16 Dual SIM, 128GB, 4G...",Samsung,Android Phones,128.0,4.0,4G,11599.0,12717.0,...,1,False,True,False,False,False,202698.0,https://www.jumia.com.eg/samsung-samsung-galax...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,2026-10-24 07:16:54


<div dir="rtl">

## الخطوة 12: فحص الداتا النضيفة

قبل الحفظ ببص بصة أخيرة:
- `df.dtypes`: نوع كل عمود. الأسعار والمساحة لازم تبقى أرقام (`float64`).
- `df.isnull().sum()`: الخانات الفاضية.
- `df["price"].describe()`: أقل وأعلى ومتوسط سعر. لو أقل سعر طلع 200 جنيه مثلاً، يبقى لسه فيه جراب أو إكسسوار.
- بتأكد إن مفيش منتج سعره الحالي أعلى من سعره القديم.

</div>

In [15]:
print(df.dtypes)
print()
print(df.isnull().sum())

site                     str
sku                      str
product_name             str
brand                    str
category                 str
storage_gb           float64
ram_gb               float64
network                  str
price                float64
old_price            float64
discount             float64
rating               float64
reviews_count          int64
is_official_store       bool
is_express              bool
is_sponsored            bool
is_renewed              bool
has_gift                bool
seller_id            float64
url                      str
image_url                str
scraped_at               str
dtype: object

site                    0
sku                     0
product_name            0
brand                   0
category                0
storage_gb             18
ram_gb                157
network               155
price                   0
old_price             236
discount              236
rating               1437
reviews_count           0
is_offici

In [16]:
print(df["price"].describe())
print()
print("Price higher than old price:", (df["price"] > df["old_price"]).sum())

count      1829.000000
mean      24492.749229
std       23860.976660
min         449.000000
25%       11099.000000
50%       15750.000000
75%       28145.000000
max      189777.000000
Name: price, dtype: float64

Price higher than old price: 0


<div dir="rtl">

## الخطوة 13: حفظ الداتا النضيفة

بحفظ في `data/clean` بنفس تاريخ ملف الـ raw، عشان أعرف كل ملف نضيف جاي من أنهي يوم.

`raw_file.replace("raw", "clean")` بتحول `jumia_raw_2026-10-24.csv` لـ `jumia_clean_2026-10-24.csv`.

</div>

In [17]:
os.makedirs("data/clean", exist_ok=True)

csv_path = "data/clean/" + raw_file.replace("raw", "clean")
excel_path = csv_path.replace(".csv", ".xlsx")

df.to_csv(csv_path, index=False, encoding="utf-8-sig")
df.to_excel(excel_path, index=False)

print("Saved:", csv_path)
print("Saved:", excel_path)

Saved: data/clean/jumia_clean_2026-10-24.csv
Saved: data/clean/jumia_clean_2026-10-24.xlsx


<div dir="rtl">

## نظرة سريعة على الداتا بعد التنظيف

دلوقتي بقيت أقدر أعمل حسابات مكانتش ممكنة والسعر نص. مثلاً متوسط سعر كل ماركة:

</div>

In [18]:
df.groupby("brand")["price"].agg(["count", "mean", "min", "max"]).sort_values("count", ascending=False).head(10).round(0)

,count,mean,min,max
brand,,,,
Samsung,459,24699.0,999.0,86199.0
Xiaomi,237,16698.0,4481.0,53990.0
Honor,237,21166.0,2679.0,90899.0
Infinix,201,12713.0,3979.0,35950.0
OPPO,177,21823.0,5559.0,55039.0
Apple,162,80074.0,20999.0,189777.0
realme,157,14930.0,4489.0,39990.0
vivo,95,17349.0,7099.0,52990.0
Tecno,49,9815.0,1767.0,21999.0


<div dir="rtl">

## الخلاصة

في الـ Notebook ده:

1. نضفت أسماء المنتجات من المسافات الزيادة.
2. حولت السعر والسعر القديم والخصم من نص لأرقام.
3. حولت التقييم `0` لـ "مفيش تقييم"، عشان ميبوظش المتوسط.
4. شلت المنتجات اللي مش موبايلات (تكييفات، أجهزة مطبخ، جرابات).
5. وحدت أسماء الماركات، وفصلت الموبايلات المستعملة في عمود `is_renewed`.
6. طلعت المساحة والرام ونوع الشبكة من اسم المنتج.
7. علمت على الموبايلات اللي معاها هدية في عمود `has_gift`.
8. حفظت الداتا النضيفة في `data/clean`.

**الخطوة الجاية:** `05_load_sql.ipynb`، تحميل الداتا النضيفة في SQLite.

</div>